# Intermediate 13 — Capstone: Secure Agent Identity & Authorization Architecture

Build a governed claims agent end to end. The LLM is an **untrusted planner**; trusted authority comes from identity, delegation, policy and enforcement.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime,timedelta,timezone
from typing import Any
import hashlib,json,uuid,copy
import pandas as pd
import networkx as nx
NOW=datetime.now(timezone.utc)


## 1 — Security invariants

In [ ]:
INVARIANTS=["model_identity_untrusted","workload_agent_binding","tenant_isolation","delegation_attenuation","pep_coverage","approval_binding","mcp_audience_binding","deny_has_no_side_effect"]
INVARIANTS


## 2 — Human, agent and workload identities

In [ ]:
human={"sub":"user:alice","tenant":"acme","assurance":2}
agent={"id":"agent:claims","registered":True,"approved_tools":{"claims.read","claims.update","knowledge.search"}}
workload={"spiffe_id":"spiffe://corp.example/prod/claims-agent","approved":True,"agent_id":"agent:claims"}
assert workload["approved"] and workload["agent_id"]==agent["id"]


## 3 — Trusted context

In [ ]:
@dataclass(frozen=True)
class SecurityContext:
    principal_id:str; tenant_id:str; agent_id:str; workload_id:str; task_id:str; delegation_id:str
ctx=SecurityContext(human["sub"],human["tenant"],agent["id"],workload["spiffe_id"],"task:483","del:483")
ctx


## 4 — OAuth audience validation

In [ ]:
def validate_token(t,issuer,audience,now):
    if t["iss"]!=issuer:return False,"ISSUER"
    if t["aud"]!=audience:return False,"AUDIENCE"
    if t["exp"]<=now:return False,"EXPIRED"
    return True,"VALID"
token={"iss":"https://id.corp","aud":"https://claims-api","exp":NOW.timestamp()+300}
validate_token(token,"https://id.corp","https://claims-api",NOW.timestamp())


## 5 — Delegation

In [ ]:
delegation={"delegator":"user:alice","delegatee":"agent:claims","tenant":"acme",
"actions":{"claim.read","claim.update","knowledge.search"},"resources":{"claim:483","kb:claims"},
"active":True,"redelegable":True,"max_depth":1,"expires_at":NOW+timedelta(hours=1)}


## 6 — Attenuate for a sub-agent

In [ ]:
def attenuate(p,delegatee,actions,resources):
    if not set(actions).issubset(p["actions"]):raise PermissionError("ACTION_ESCALATION")
    if not set(resources).issubset(p["resources"]):raise PermissionError("RESOURCE_ESCALATION")
    return {"delegator":p["delegatee"],"delegatee":delegatee,"tenant":p["tenant"],
            "actions":set(actions),"resources":set(resources),"active":True,
            "expires_at":p["expires_at"]}
child=attenuate(delegation,"agent:research",{"claim.read","knowledge.search"},{"claim:483","kb:claims"})
child


## 7 — Typed intent

In [ ]:
@dataclass
class Intent:
    action:str; resource:str; tool:str; purpose:str; parameters:dict[str,Any]=field(default_factory=dict)
intent=Intent("claim.update","claim:483","claims.update","process claim",{"status":"reviewed"})


## 8 — Rich policy decision

In [ ]:
@dataclass
class Decision:
    outcome:str; decision_id:str; reason:str; constraints:dict=field(default_factory=dict); obligations:list=field(default_factory=list)


## 9 — PDP

In [ ]:
def authorize(ctx,intent,d,a,w):
    deny=lambda r:Decision("deny",uuid.uuid4().hex,r)
    if not a["registered"]:return deny("AGENT_UNREGISTERED")
    if not w["approved"] or w["agent_id"]!=ctx.agent_id:return deny("WORKLOAD_BINDING")
    if d["tenant"]!=ctx.tenant_id:return deny("TENANT_MISMATCH")
    if d["delegatee"]!=ctx.agent_id:return deny("WRONG_DELEGATEE")
    if not d["active"] or d["expires_at"]<=NOW:return deny("DELEGATION_INACTIVE")
    if intent.action not in d["actions"]:return deny("ACTION_OUT_OF_SCOPE")
    if intent.resource not in d["resources"]:return deny("RESOURCE_OUT_OF_SCOPE")
    c={"allowed_fields":{"status","notes"}} if intent.action=="claim.update" else {}
    return Decision("allow",uuid.uuid4().hex,"TASK_SCOPE",c,["audit"])
decision=authorize(ctx,intent,delegation,agent,workload)
decision


## 10 — Enforce decision constraints

In [ ]:
def enforce(intent,d):
    if d.outcome!="allow":raise PermissionError(d.reason)
    f=d.constraints.get("allowed_fields")
    if f is not None and not set(intent.parameters).issubset(f):raise PermissionError("FIELD_CONSTRAINT")
    return True
enforce(intent,decision)


## 11 — Cross-tenant and workload attacks

In [ ]:
evil=copy.deepcopy(delegation);evil["tenant"]="other"
print(authorize(ctx,intent,evil,agent,workload))
bad={**workload,"agent_id":"agent:attacker"}
print(authorize(ctx,intent,delegation,agent,bad))


## 12 — Authorization-aware RAG

In [ ]:
docs=[{"id":"d1","tenant":"acme","acl":{"claims"}},{"id":"d2","tenant":"other","acl":{"claims"}},{"id":"d3","tenant":"acme","acl":{"hr"}}]
groups={"claims"}
[x for x in docs if x["tenant"]==ctx.tenant_id and x["acl"] & groups]


## 13 — Memory authorization

In [ ]:
mem=[{"id":"m1","tenant":"acme","owner":"user:alice"},{"id":"m2","tenant":"acme","owner":"user:bob"}]
[x for x in mem if x["tenant"]==ctx.tenant_id and x["owner"]==ctx.principal_id]


## 14 — Multi-agent authority graph

In [ ]:
g=nx.DiGraph()
g.add_edge("user:alice","agent:claims",actions=delegation["actions"])
g.add_edge("agent:claims","agent:research",actions=child["actions"])
list(g.edges(data=True))


## 15 — MCP audience binding

In [ ]:
mcp_token={"aud":"https://mcp.claims.example","scope":{"claims.read"}}
assert mcp_token["aud"]=="https://mcp.claims.example"
print("Do not forward this token unchanged to https://claims-api.internal")


## 16 — Risk and HITL

In [ ]:
def risk(i):
    if i.action=="payment.create":
        x=float(i.parameters.get("amount",0))
        return "critical" if x>10000 else "high" if x>500 else "medium"
    return "low"
payment=Intent("payment.create","account:42","payments.create","settle claim",{"amount":750,"currency":"CAD"})
risk(payment)


## 17 — Bind approval to transaction

In [ ]:
def digest(ctx,i):
    x={"principal":ctx.principal_id,"agent":ctx.agent_id,"task":ctx.task_id,"action":i.action,"resource":i.resource,"tool":i.tool,"parameters":i.parameters}
    return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(",",":")).encode()).hexdigest()
approval={"approver":"user:manager","digest":digest(ctx,payment),"expires_at":NOW+timedelta(minutes=5)}
changed=copy.deepcopy(payment);changed.parameters["amount"]=7500
assert digest(ctx,changed)!=approval["digest"]


## 18 — Evidence

In [ ]:
def evidence(ctx,i,d,result):
    return {"timestamp":NOW.isoformat(),"trace_id":uuid.uuid4().hex,"task_id":ctx.task_id,
    "principal_id":ctx.principal_id,"agent_id":ctx.agent_id,"workload_id":ctx.workload_id,
    "delegation_id":ctx.delegation_id,"action":i.action,"resource":i.resource,"tool":i.tool,
    "decision_id":d.decision_id,"decision":d.outcome,"reason":d.reason,"result":result}
evidence(ctx,intent,decision,"success")


## 19 — PEP coverage

In [ ]:
paths=pd.DataFrame([["agent→router→api",True],["agent→mcp→api",True],["queue→worker→api",True],["agent→api-direct",False]],columns=["path","pep"])
paths


## 20 — Revocation and stale cache

In [ ]:
revoked=copy.deepcopy(delegation);revoked["active"]=False
authorize(ctx,intent,revoked,agent,workload)


## 21 — Attack corpus

In [ ]:
attacks=["identity_spoofing","wrong_audience","workload_impersonation","delegation_escalation","cross_tenant_idor","pep_bypass","fail_open","stale_cache","parameter_swap","mcp_substitution","authority_laundering","toctou"]
pd.DataFrame({"attack":attacks})


## 22 — Mutation suite

In [ ]:
mutations=["remove_tenant_check","remove_workload_binding","wildcard_resource","disable_audience_validation","remove_approval_requirement","unbounded_redelegation"]
mutations


## 23 — Architecture scorecard

In [ ]:
pd.DataFrame([
["Human identity","PASS"],["Agent registration","PASS"],["Workload binding","PASS"],
["Delegation attenuation","PASS"],["Tenant isolation","PASS"],["Resource authorization","PASS"],
["MCP audience binding","PASS"],["Approval binding","PASS"],["PEP path coverage","FAIL"],
["Evidence correlation","PASS"]],columns=["control","status"])


## 24 — OPA, Cedar and OpenFGA lab

Use the included policy files. Run the same positive and adversarial corpus against each approach. Compare relationship modeling, contextual policy, testability, operational model and audit evidence. Explain what each engine should own rather than trying to force all authorization into one abstraction.


## 25 — OpenAI Agents SDK / LangGraph integration

Replace the simulated planner with a real framework. With OpenAI Agents SDK, use function tools, tool guardrails, HITL interruptions, MCP tool filtering/approval and tracing. With LangGraph, make `authorize` an explicit node before every consequential execution node.

Keep authorization deterministic and external to model judgment.


## 26 — Final review

For every arrow in the architecture answer: identity, authentication, delegated authority, resource, PDP, PEP, failure behavior, revocation, evidence, and adversarial test.

### Final deliverables

- architecture + trust-boundary diagrams
- identity inventory and agent registry
- workload mapping
- delegation model
- OPA/Cedar/OpenFGA examples
- secure tool dispatcher
- MCP authorization design
- RAG/memory ACL design
- HITL transaction binding
- evidence schema
- attack + mutation suites
- PEP coverage report
- CI/CD gates
- revocation/failure runbook
